# 02 — Language Models: Transformer · Mamba · RNN · LoRA · RAG

This notebook covers all text models, from classic RNNs to SSMs and
retrieval-augmented generation.

## Architecture overview

```
  Transformer (decoder-only)
  Token → Embed → [Attn + FFN] × N → LM head → next token
  FFN can be: Standard | MoE (gated experts) | Mamba (SSM)

  NanoMamba (pure SSM)
  Token → Embed → [MambaBlock] × N → LM head
  MambaBlock: depthwise Conv1d + gated exponential decay scan

  RNN / LSTM / GRU
  Token → Embed → cell (recurrent hidden state) → LM head
  Sequential: one token at a time, carries hidden state

  LoRA fine-tuning
  W·x  +  (x @ A^T @ B^T) × (α/r)
  Only A, B are updated → much fewer parameters

  RAG
  Query → TF-IDF retriever → context → LM → response
```

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_JUPYTER_TOKEN'))
if IN_COLAB:
    !pip install -e . -q

sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))
from viz import plot_metrics, show_text_sample, show_lora_deltas
from mini_networks.colab.launcher import run_model

---
## 1. Transformer LM

Standard decoder-only Transformer trained on Tiny Shakespeare with char-level tokenisation.

In [ ]:
tr_logger = run_model('transformer', fast_demo=True)

In [ ]:
plot_metrics(tr_logger, keys=['loss'])

In [ ]:
from mini_networks.models.transformer.config import TransformerConfig
from mini_networks.models.transformer.trainer import TransformerTrainer

config = TransformerConfig(fast_demo=True)
trainer = TransformerTrainer()
trainer.load_checkpoint(config, tr_logger.artifacts_dir)

result = trainer.infer(config, {'prompt': 'HAMLET:', 'max_new_tokens': 80, 'temperature': 0.8})
show_text_sample(result.get('generated', str(result)), title='Transformer generation')

---
## 2. NanoMamba (pure SSM)

State-space model alternative to attention — linear time complexity with no quadratic attention.

In [ ]:
mamba_logger = run_model('mamba', fast_demo=True)

In [ ]:
plot_metrics(mamba_logger, keys=['loss'])

In [ ]:
from mini_networks.models.mamba.config import MambaConfig
from mini_networks.models.mamba.trainer import MambaTrainer

config = MambaConfig(fast_demo=True)
trainer = MambaTrainer()
trainer.load_checkpoint(config, mamba_logger.artifacts_dir)

result = trainer.infer(config, {'max_new_tokens': 60, 'temperature': 0.9})
show_text_sample(result.get('generated', str(result)), title='NanoMamba generation')

---
## 3. RNN / LSTM / GRU

Classic recurrent architectures — sequential by design, but efficient at inference
with explicit hidden state carry-over.

In [ ]:
rnn_logger = run_model('rnn', fast_demo=True)

In [ ]:
plot_metrics(rnn_logger, keys=['loss'])

In [ ]:
from mini_networks.models.rnn.config import RNNConfig
from mini_networks.models.rnn.trainer import RNNTrainer

config = RNNConfig(fast_demo=True)
trainer = RNNTrainer()
trainer.load_checkpoint(config, rnn_logger.artifacts_dir)

result = trainer.infer(config, {'max_new_tokens': 60, 'temperature': 0.9})
show_text_sample(result.get('generated', str(result)), title='RNN generation')

---
## 4. LoRA fine-tuning

Two-stage: pre-train CNN on MNIST (full params), fine-tune on FashionMNIST (LoRA A,B only).

> **Why LoRA?** Adapts a frozen pre-trained model to a new task by updating only
> low-rank decomposition matrices — far fewer parameters, same expressivity.

In [ ]:
lora_logger = run_model('lora', fast_demo=True)

In [ ]:
plot_metrics(lora_logger, keys=['pretrain_loss', 'finetune_loss'])

In [ ]:
from mini_networks.models.lora.config import LoRAConfig
from mini_networks.models.lora.trainer import LoRATrainer
import torch

config = LoRAConfig(fast_demo=True)
trainer = LoRATrainer()
trainer.load_checkpoint(config, lora_logger.artifacts_dir)

# LoRA weight delta norms — bar chart of trainable LoRA params
show_lora_deltas(trainer, title='LoRA A/B weight norms after fine-tuning')

---
## 5. RAG — Retrieval-Augmented Generation

Retrieves relevant chunks from the corpus via TF-IDF cosine similarity,
prepends them to the prompt, then generates with the Transformer LM.

In [ ]:
rag_logger = run_model('rag', fast_demo=True)

In [ ]:
plot_metrics(rag_logger, keys=['loss'])

In [ ]:
# RAG inference — note: load_checkpoint() doesn't rebuild the TF-IDF index;
# use the in-memory trainer from run_model() for full RAG inference.
# The trainer is accessible via run_model(return_trainer=True) if your launcher
# supports it, or rebuild via:
from mini_networks.models.rag.config import RAGConfig
from mini_networks.models.rag.trainer import RAGTrainer, make_rag_dataloader
from mini_networks.core.logging.logger import Logger

config = RAGConfig(fast_demo=True)
trainer = RAGTrainer()
import tempfile, os
with tempfile.TemporaryDirectory() as tmp:
    logger = Logger(tmp, 'rag_demo')
    dl = make_rag_dataloader(config)
    trainer.train(config, dl, logger)  # rebuilds model + RAG index in memory

result = trainer.infer(config, {'query': 'To be or not to be', 'max_new_tokens': 50})
print('Retrieved context snippet:')
print(result['retrieved'][:200])
show_text_sample(result['generated'], title='RAG-augmented generation')